### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
%pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part3_llm_training_tests.py

import mugrade
import os
from part3_llm_training_tests import *

def _mugrade_name(name):
    def rename(function):
        function.__name__ = name
        return function
    return rename

os.environ["MUGRADE_HW"] = "Part 3 - LLM Training"
# Read the Mugrade API key from the environment or from a local .env file.
# Never hard-code the key. See .env.example in the repository root.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
if not os.environ.get("MUGRADE_KEY"):
    print("MUGRADE_KEY is not set. Cells that use @mugrade.submit_tests need it.")

### BPE

Insert the BPE tokenizer you built in Part 1.

In [2]:
### BEGIN YOUR CODE
import json
from collections import Counter
import re
from tqdm.auto import tqdm

class BPE:
    def __init__(self, filename=None):
        ### BEGIN YOUR CODE
        if filename is not None:
            with open(filename, "rt") as fobj:
                self.vocab, merge_keys, merge_values, self.special_tokens = json.load(fobj)
                self.merges = {
                    tuple(k): v for k, v in zip(merge_keys, merge_values)
                }
                self.special_tokens = {
                    int(k): v for k, v in self.special_tokens.items()
                }
        else:
            self.vocab = [chr(i) for i in range(256)]
            self.merges = {}
            self.special_tokens = {}
        ### END YOUR CODE

    @staticmethod
    def merge_pair(word, a, b, merged):
        ### BEGIN YOUR CODE
        i = 0
        while i < len(word) - 1:
            if word[i] == a and word[i+1] == b:
                word[i: i+2] = [merged]
            i+=1
        ### END YOUR CODE

    def replace_special_tokens(self, word):
        ### BEGIN YOUR CODE
        for k, v in self.special_tokens.items():
            word = word.replace(v, chr(k))
        return word
        ### END YOUR CODE


    def train(self, text, target_vocab_size):
        ### BEGIN YOUR CODE
        regex_matching = re.findall(r"\s\S+|\s+|\S+", text)
        frequency_counter = Counter(regex_matching)
        frequency_map = {
            self.replace_special_tokens(k): v for k, v in frequency_counter.items() if v > 1
        }

        # Build word_list_frequency_map without defaultdict
        word_list_frequency_map = {}
        
        for word, freq in frequency_map.items():
            filtered_word = tuple(
                ord(char) for char in word 
                if ord(char) not in self.special_tokens
            )
            
            if filtered_word:
                word_list_frequency_map[filtered_word] = \
                    word_list_frequency_map.get(filtered_word, 0) + freq

        words_as_lists, words_as_lists_frequencies = \
            [list(i) for i in word_list_frequency_map.keys()], list(word_list_frequency_map.values())


        for _ in range(len(self.vocab), target_vocab_size):
            existing_vocab_size = len(self.vocab)
            suitable_merge_pairs = Counter()

            for current_word_as_list, current_word_frequency in \
                zip(words_as_lists, words_as_lists_frequencies):

                for i in range(len(current_word_as_list) - 1):
                    suitable_merge_pairs[(current_word_as_list[i], current_word_as_list[i+1])] +=\
                        current_word_frequency

            # Break condition for testing
            # if len(suitable_merge_pairs) == 0:
            #     break

            symbol_a, symbol_b = suitable_merge_pairs.most_common()[0][0]
            self.vocab.append(self.vocab[symbol_a] + self.vocab[symbol_b])
            self.merges[(symbol_a, symbol_b)] = existing_vocab_size

            for word_as_list in words_as_lists:
                self.merge_pair(word_as_list, symbol_a, symbol_b, existing_vocab_size)

        if len(self.special_tokens) != 0:
            max_special_token = max(self.special_tokens.keys())
            self.vocab += [""] * (max_special_token - len(self.vocab) + 1)
            for k, v in self.special_tokens.items():
                self.vocab[k] = v
        ### END YOUR CODE

    def encode(self, text):
        ### BEGIN YOUR CODE
        tokens = []

        for word in re.findall(r"\s\S+|\s+|\S+", text):
            word_as_list = [ord(char) for char in self.replace_special_tokens(word)]
            # Special tokens are now single code points (e.g. chr(32760)). No merge
            # contains them, so they pass through as their own token IDs.
            
            if  len(word_as_list) == 0:
                continue

            while True:
                word_symbol_pairs_to_vocab_index_mapping = {}
                for i in range(len(word_as_list) - 1):
                    current_symbol_pair = \
                        (word_as_list[i], word_as_list[i+1])
                    if current_symbol_pair in self.merges:
                        word_symbol_pairs_to_vocab_index_mapping[current_symbol_pair] = \
                            self.merges[current_symbol_pair]

                if len(word_symbol_pairs_to_vocab_index_mapping) == 0:
                    break

                symbol_a, symbol_b = min_symbol_pair = min(
                    word_symbol_pairs_to_vocab_index_mapping,
                    key = word_symbol_pairs_to_vocab_index_mapping.get
                )
                self.merge_pair(word_as_list, symbol_a, symbol_b, 
                                word_symbol_pairs_to_vocab_index_mapping[min_symbol_pair])

            tokens += word_as_list

        return tokens
        ### END YOUR CODE

    def decode(self, tokens):
        ### BEGIN YOUR CODE
        return "".join([self.vocab[token] for token in tokens])
        ### END YOUR CODE

    def save(self, fname):
        ### BEGIN YOUR CODE
        with open(fname, "wt") as fobj:
            json.dump([
                self.vocab,
                list(self.merges.keys()),
                list(self.merges.values()),
                self.special_tokens
            ], fobj)
        ### END YOUR CODE
### END YOUR CODE

### LLM Architecture

Insert your LLM architecture from Part 2.

In [3]:
### BEGIN YOUR CODE
import torch
import math

def embedding(x, weights, dtype):
    ### BEGIN YOUR CODE
    return weights[x].to(dtype=dtype)
    ### END YOUR CODE

def linear(x, weights):
    ### BEGIN YOUR CODE
    return x @ weights.type_as(x)
    ### END YOUR CODE

def silu(x):
    ### BEGIN YOUR CODE
    return x / (1 + torch.exp(-x))
    ### END YOUR CODE

def rms_norm(x):
    ### BEGIN YOUR CODE
    x_typed = x.float()
    x_normed = x_typed / (
        (x_typed**2)
            .mean(
                dim = -1,
                keepdim = True,
            )
            .sqrt()
    )
    return x_normed.type_as(x)
    ### END YOUR CODE

def softmax(x):
    ### BEGIN YOUR CODE
    x_numerical_stable = \
        x - x.max(dim = -1, keepdim=True).values
    x_exp = x_numerical_stable.exp()
    return x_exp / (
        x_exp.sum(dim=-1, keepdim=True)
    )
    ### END YOUR CODE

def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    attention_score = softmax(
        q @ k.transpose(-2, -1) / math.sqrt(k.size(-1)) + mask
    )
    return attention_score @ v
    ### END YOUR CODE

class LLM:
    def rope(self, x):
        ### BEGIN YOUR CODE
        return (
            (self.buffers["rope_w1"] * x) + 
            (self.buffers["rope_w2"] * x.reshape(-1, 
                2).flip(-1).reshape_as(x))
        ).type_as(x)
        ### END YOUR CODE

    def multihead_attn(self, x, layer, mask):
        ### BEGIN YOUR CODE
        
        B, L, D = x.shape
        q = linear(x, self.params[f"wq_{layer}"])
        k = linear(x, self.params[f"wk_{layer}"])
        v = linear(x, self.params[f"wv_{layer}"])
        wp = self.params[f"wp_{layer}"]

        q = q.reshape(
            B, L, self.num_heads, D // self.num_heads
        ).transpose(1, 2)
        k = k.reshape(
            B, L, self.num_heads, D // self.num_heads
        ).transpose(1, 2)
        v = v.reshape(
            B, L, self.num_heads, D // self.num_heads
        ).transpose(1, 2)

        q, k = rms_norm(self.rope(q)), rms_norm(self.rope(k))

        attn_output = self_attn(q,k,v,mask)

        return linear(
            attn_output.transpose(1, 2).reshape_as(x),
            wp
        )

        ### END YOUR CODE

    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        return linear(
            silu(
                linear(
                    x,
                    self.params[f"w1_{layer}"]
                )
            ),
            self.params[f"w2_{layer}"]
        )
        ### END YOUR CODE

    def transformer_block(self, x, layer, mask):
        ### BEGIN YOUR CODE
        x = x + \
            self.multihead_attn(
                rms_norm(x), layer, mask
            )
        return x + self.mlp(
            rms_norm(x), layer
        )
        ### END YOUR CODE

    def __init__(self, config):
        ### BEGIN YOUR CODE
        self.num_layers = config["depth"]
        self.dtype = config["dtype"]

        vocab_size = config["vocab_size"]
        sequence_len = config["seq_len"]

        hidden_dim = self.num_layers * config["aspect_ratio"]
        self.mlp_dim = hidden_dim * config["mlp_multiple"]

        self.head_dim = config["head_dim"]
        self.num_heads = hidden_dim // self.head_dim

        self.params = {}
        dim_scale, dim_scale_mlp = \
            math.sqrt(2/hidden_dim), math.sqrt(2/self.mlp_dim)

        self.params["embedding"] = torch.randn(
            vocab_size, hidden_dim
        ) * dim_scale
        self.params["output"] = torch.randn(
            hidden_dim, vocab_size
        ) * dim_scale

        for i in range(self.num_layers):
            self.params[f"wq_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wk_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wv_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"wp_{i}"] = \
                torch.randn(
                    hidden_dim, hidden_dim
                ) * dim_scale
            self.params[f"w1_{i}"] = \
                torch.randn(
                    hidden_dim, self.mlp_dim
                ) * dim_scale
            self.params[f"w2_{i}"] = \
                torch.randn(
                    self.mlp_dim, hidden_dim
                ) * dim_scale_mlp

        self.buffers = {}
        mask = torch.full(
            (sequence_len, sequence_len),
            -float("inf"))
        self.buffers["mask"] = \
            torch.triu(mask, diagonal=1)


        freqs = config["rope_theta"] ** (
            (-2 * torch.arange(
                self.head_dim // 2
            ) / self.head_dim)
        )
        theta = torch.outer(
            torch.arange(sequence_len),
            freqs
        )

        cos, sin = torch.cos(theta), torch.sin(theta)
        self.buffers["rope_w1"] = \
            torch.stack([cos, cos], dim = -1).reshape(
                sequence_len, self.head_dim
            )
        self.buffers["rope_w2"] = \
            torch.stack([-sin, sin], dim = -1).reshape(
                sequence_len, self.head_dim
            )

        ### END YOUR CODE

    def __call__(self, tokens):
        ### BEGIN YOUR CODE
        x = rms_norm(
            embedding(
                tokens,
                self.params["embedding"],
                self.dtype
            )
        )

        for i in range(self.num_layers):
            x = self.transformer_block(
                x, i, self.buffers["mask"]
            )

        x_final = \
            rms_norm(x)

        return linear(
            x_final, 
            self.params["output"]
        )
        ### END YOUR CODE

    def save(self, filename):
        ### BEGIN YOUR CODE
        torch.save(self.params, filename)
        ### END YOUR CODE

    def load(self, filename):
        ### BEGIN YOUR CODE
        self.params = \
            torch.load(filename, weights_only=True)
        ### END YOUR CODE

### END YOUR CODE

### Downloading necessary files

These next lines will donwload the necessary tokenizer and pre-tokenized data files, which you can build in Part 1 (but which require a fairly substantial machine to run as-is).

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo = "zkolter/llm_speedrun"
filenames = ["fineweb-edu-10BT.shuffle.bin", "smoltalk.shuffle.bin", "tokenizer_50M.bpe"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=".")

You can use the following config file for training.  This is intended to load a batch size that will fit comfortably on a GPU with 80GB of memory, you can adjust as needed.

In [ ]:
%%writefile config_original.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 16,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 7e-4,
    "weight_decay": 0.025,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 1
}

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 4,
    "seq_len": 1024,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 1.5e-4,
    "weight_decay": 0.025,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 1
}

### LLM Training

In [ ]:
%pip install wandb

In [ ]:
import wandb
import time
from array import array
import os


@mugrade.submit_tests
def cross_entropy_loss(logits, y):
    ### BEGIN YOUR CODE
    logits_normed = logits - logits.max(dim = -1, keepdim = True)[0]
    loss = -logits_normed.take_along_dim(y.unsqueeze(-1), dim = -1)
    loss += logits_normed.exp().sum(dim = -1, keepdim = True).log()
    return loss.mean()
    ### END YOUR CODE

class Adam:
    @mugrade.submit_tests
    @_mugrade_name("Adam_init")
    def __init__(self, params, lr_schedule, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        self.schedule = lr_schedule
        self.betas = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.params = params

        self.t = 1

        self.u = {k: torch.zeros_like(v) for k,v in self.params.items()}
        self.v = {k: tensor.clone().detach() for k, tensor in self.u.items()}

        for param in self.params.values():
            param.requires_grad_()
        ### END YOUR CODE

    @mugrade.submit_tests
    def step(self):
        ### BEGIN YOUR CODE
        with torch.no_grad():
            for k,p in self.params.items():
                self.u[k] = self.betas[0]*self.u[k] + (1-self.betas[0])*p.grad
                self.v[k] = self.betas[1]*self.v[k] + (1-self.betas[1])*p.grad**2

                u_hat = self.u[k] / (1 - self.betas[0]**self.t)
                v_hat = self.v[k] / (1 - self.betas[1]**self.t)
                
                p *= (1 - self.schedule.get_lr(self.t) * self.weight_decay)
                p -= self.schedule.get_lr(self.t) * u_hat / (v_hat.sqrt() + self.eps)

                p.grad.zero_()
            self.t += 1
        ### END YOUR CODE

class LRSchedule:
    @mugrade.submit_tests
    @_mugrade_name("LRSchedule_init")
    def __init__(self, total_steps, lr=1e-3, warmup_steps=50, decay_ratio=0.4, min_frac=0.1):
        ### BEGIN YOUR CODE
        self.total_steps = total_steps
        self.lr = lr
        self.warmup_steps = warmup_steps
        self.decay_ratio = decay_ratio
        self.min_frac = min_frac

        self.decay_steps = round(self.decay_ratio * self.total_steps)
        ### END YOUR CODE

    @mugrade.submit_tests
    def get_lr(self, step):
        ### BEGIN YOUR CODE
        if step < self.warmup_steps:
            ratio = step / self.warmup_steps
            return ratio * self.lr + (1 - ratio) * self.min_frac * self.lr
        elif step > self.total_steps - self.decay_steps:
            ratio = (step - (
                self.total_steps - self.decay_steps
            )) / self.decay_steps
            return (1 - ratio) * self.lr + ratio * self.lr * self.min_frac
        else:
            return self.lr
        ### END YOUR CODE


@mugrade.submit_tests
def train_llm(config, log=False):
    ### BEGIN YOUR CODE
    llm = LLM(config)
    tokenizer = BPE(config["tokenizer"])

    # Move to cuda
    for k in llm.params: 
        llm.params[k] = llm.params[k].cuda()
    for k in llm.buffers: 
        llm.buffers[k] = llm.buffers[k].cuda()
    
    config["total_params"] = sum(p.numel() for p in llm.params.values())
    config["total_tokens"] = config["total_params"] * config["token_multiple"]
    config["total_steps"] = config["total_tokens"] / (config["batch_size"] * config["seq_len"])

    schedule = LRSchedule(config["total_steps"], config["lr"])
    opt = Adam(llm.params, schedule, weight_decay=config["weight_decay"])

    if log:
        run = wandb.init(project="llm_speedrun", config=config)
        
    batch_items = []
    for k,p in config["data_mix"].items():
        batch_items += [(k, i*(config["seq_len"]+1)*2) for i in range(round(p*config["batch_size"]))]

    file_offsets = {k:0 for k in config["data_mix"]}
    n_tok = 0
    
    while opt.t < config["total_steps"]:
        start_time = time.perf_counter()

        tokens = []

        for filename, offset in batch_items:
            with open(filename, "rb") as f:
                f.seek(file_offsets[filename] + offset)
                tokens += array("H", f.read((config["seq_len"]+1)*2)).tolist()

        for k,p in config["data_mix"].items():
            # Advance by the rows actually read (round(p * batch_size) rows), so the
            # offset stays on a uint16 boundary even when p * batch_size is fractional.
            read_size = round(p*config["batch_size"]) * (config["seq_len"]+1) * 2
            file_offsets[k] += read_size
            if file_offsets[k] + read_size >= os.path.getsize(k):
                file_offsets[k] = 0

        tokens = torch.tensor(tokens).reshape(config["batch_size"], config["seq_len"]+1)
        text = tokenizer.decode(tokens[:,1:].flatten().tolist())

        tokens = tokens.cuda()
        logits = llm(tokens[:,:-1]).float()
        loss = cross_entropy_loss(logits, tokens[:,1:])

        # Optimize
        loss.backward()
        opt.step()

        # Logging
        bpb = loss.item() * config["batch_size"] * config["seq_len"] / (len(text) * math.log(2))
        n_tok += config["batch_size"] * config["seq_len"]
        tok_per_sec = config["batch_size"] * config["seq_len"] / (time.perf_counter() - start_time)

        if log:
            run.log({
                "step": opt.t,
                "tokens": n_tok,
                "loss": loss.item(),
                "bpb": bpb,
                "lr": opt.schedule.get_lr(opt.t),
                "tok_per_sec": tok_per_sec})

    return llm
    ### END YOUR CODE

If your code passes all the tests, you can (optionally) uncomment this block to train the d12 model, here done on a single GPU.

In [ ]:
with open("config.d12.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]

llm = train_llm(config, log=True)

### Distributed LLM Training

Implement a distributed version of the training code, starting from the version above.  Note that there was one error in the class version, that `config["batch_size"]` should be replaced by `local_batch_size` in the line that computes bpb for logging.

In [26]:
@mugrade.submit_tests
def train_llm_distributed(rank, nccl_uid, config, log=False):
    ### BEGIN YOUR CODE
    torch.cuda.set_device(rank)

    llm = LLM(config)
    tokenizer = BPE(config["tokenizer"])

    # Move to cuda
    for k in llm.params: 
        llm.params[k] = llm.params[k].cuda()
    for k in llm.buffers: 
        llm.buffers[k] = llm.buffers[k].cuda()

    # Distributed comms
    comm = torch.cuda.nccl.init_rank(config["num_gpus"], nccl_uid, rank)
    for p in llm.params.values():
        torch.cuda.nccl.broadcast([p], root=0, comms=[comm])
    
    config["total_params"] = sum(p.numel() for p in llm.params.values())
    config["total_tokens"] = config["total_params"] * config["token_multiple"]
    config["total_steps"] = config["total_tokens"] / (config["batch_size"] * config["seq_len"])

    schedule = LRSchedule(config["total_steps"], config["lr"])
    opt = Adam(llm.params, schedule, weight_decay=config["weight_decay"])
    
    if log and rank == 0:
        run = wandb.init(project="llm_speedrun", config=config)

    batch_items = []
    for k,p in config["data_mix"].items():
        batch_items += [(k, i*(config["seq_len"]+1)*2) for i in range(round(p*config["batch_size"]))]

    file_offsets = {k:0 for k in config["data_mix"]}
    n_tok = 0

    local_batch_size = config["batch_size"] // config["num_gpus"]
    
    while opt.t < config["total_steps"]:
        start_time = time.perf_counter()

        tokens = []

        for filename, offset in batch_items[
            int(rank*local_batch_size):int((rank + 1)*local_batch_size)
        ]:
            with open(filename, "rb") as f:
                f.seek(file_offsets[filename] + offset)
                tokens += array("H", f.read((config["seq_len"]+1)*2)).tolist()

        for k,p in config["data_mix"].items():
            # Advance by the rows actually read (round(p * batch_size) rows), so the
            # offset stays on a uint16 boundary even when p * batch_size is fractional.
            read_size = round(p*config["batch_size"]) * (config["seq_len"]+1) * 2
            file_offsets[k] += read_size
            if file_offsets[k] + read_size >= os.path.getsize(k):
                file_offsets[k] = 0

        tokens = torch.tensor(tokens).reshape(local_batch_size, config["seq_len"]+1)
        text = tokenizer.decode(tokens[:,1:].flatten().tolist())

        text_len = torch.tensor(len(text), dtype=torch.float32).cuda()

        tokens = tokens.cuda()
        logits = llm(tokens[:,:-1]).float()
        loss = cross_entropy_loss(logits, tokens[:,1:])

        # Optimize
        loss.backward()
        for p in llm.params.values():
            torch.cuda.nccl.all_reduce([p.grad], comms=[comm])
            p.grad /= config["num_gpus"]

        opt.step()

        # Logging: sum the per-rank mean losses and byte counts over all ranks.
        # Total nats = sum_r(loss_r) * local_batch_size * seq_len.
        torch.cuda.nccl.all_reduce([loss], comms=[comm])
        torch.cuda.nccl.all_reduce([text_len], comms=[comm])
        bpb = loss.item() * local_batch_size * config["seq_len"] / (text_len.item() * math.log(2))

        n_tok += config["batch_size"] * config["seq_len"]
        tok_per_sec = config["batch_size"] * config["seq_len"] / (time.perf_counter() - start_time)


        if log and rank==0:
            run.log({
                "step": opt.t,
                "tokens": n_tok,
                "loss": loss.item() / config["num_gpus"],
                "bpb": bpb, 
                "lr": opt.schedule.get_lr(opt.t),
                "tok_per_sec": tok_per_sec})

    if rank == 0:
        return llm

    ### END YOUR CODE

Mugrade: Submitting tests for function train_llm_distributed():
Grader test 1 passed
Grader test 2 passed
Grader test 3 passed


If you're able to access a machine with 8 GPUs, then the following code will launch the distributed training run with a different config that uses a larger batch size.

In [ ]:
%%writefile config.d12.8gpu.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 128,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 1e-3,
    "weight_decay": 0.1,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 8
}

In [ ]:
from joblib import Parallel, delayed
os.environ["NCCL_NVLS_ENABLE"] = "0"  # Work around an NVLink SHARP (NVLS) issue on some multi-GPU hosts.

with open("config.d12.8gpu.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]
nccl_uid = torch.cuda.nccl.unique_id()

Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(train_llm_distributed)(i, nccl_uid, config, log=True) for i in range(config["num_gpus"])
)